### oliveyoung review web crawler - global (Newest)

## 📋 크롤러 개요

이 노트북은 **Selenium(undetected_chromedriver) + BeautifulSoup**을 이용해 올리브영 글로벌(global.oliveyoung.com) 특정 상품의 리뷰 페이지에 접속한 뒤, **"더보기(Load more)" 버튼을 반복 클릭**하며 화면에 렌더링되는 리뷰 HTML을 직접 파싱해 수집하는 크롤러입니다.

### 🔍 수집 방식
1. 상품 링크(URL) 접속 → 리뷰 섹션으로 이동
2. 정렬 기준(Newest / Highest rating / Lowest rating) 버튼 클릭
3. 화면에 로드된 리뷰 HTML을 BeautifulSoup으로 파싱하여 리스트에 누적
4. 더 이상 새 리뷰가 로드되지 않을 때까지 "더보기" 버튼 클릭을 반복 (무한 로드형 페이지네이션)
5. 수집 완료 후 DataFrame으로 정리, CSV(+XLSX)로 저장

> 국내(KR) 크롤러가 리뷰 API를 직접 호출하는 방식이었다면, 글로벌(GB) 크롤러는 별도로 노출된 리뷰 API가 없어 **브라우저 화면에 실제 렌더링된 DOM을 그대로 읽어오는 방식**을 사용합니다. KR/GB 두 사이트의 구조 차이 때문에 수집 방식 자체를 다르게 설계한 부분이라, 발표자료에서 이 차이를 짚어주면 기술적 완성도가 잘 드러나요.

### 📦 수집 데이터 항목
| 컬럼명 | 설명 |
|---|---|
| goods_no | 상품 코드 |
| writer | 작성자 닉네임 |
| skin_type | 피부 타입 (Oily, Dry,,) |
| skin_tone | 피부 톤 (Porcelain, Fair, Dark,,) |
| skin_trouble | 피부 고민 (Acne, Blackheads,,) |
| rating | 평점 |
| review_date | 작성일 |
| review_txt | 리뷰 본문 |
| helpful_cnt | 추천수 (도움돼요 수) |

### ⚙️ 정렬 기준 3종 구성
이 노트북은 **세 개의 독립된 크롤러 블록**으로 구성되어 있으며, 각 블록이 서로 다른 정렬 기준으로 리뷰를 수집합니다.

1. **Newest** — 최신순
2. **Highest rating** — 평점 높은순
3. **Lowest rating** — 평점 낮은순

세 블록의 로직은 거의 동일하고, 정렬 버튼을 클릭하는 CSS 셀렉터만 다릅니다.
(`PD_reviewSortNewest_click` / `PD_reviewSortHighestRating_click` / `PD_reviewSortLowestRating_click`)
특정 정렬 기준의 리뷰만 필요하다면 해당 블록만 실행하면 됩니다.

### ✅ 실행 전 참고 및 수정이 필요한 부분
- **상품 링크(URL)를 미리 확보해두어야 합니다.** 올리브영 글로벌 상품 상세페이지 URL 형식:
  ```
  https://global.oliveyoung.com/product/detail?prdtNo=XXXXXXXXXX
  ```
  `prdtNo` 값이 상품별 고유 코드이며, 각 블록의 step 2 셀에서 직접 입력받습니다.
- 로컬에 설치된 **Chrome 브라우저 버전을 미리 확인**해두어야 합니다 (`VERSION_MAIN` 값, 3개 블록 모두 동일하게 수정 필요).
- 리뷰 카드의 **CSS 셀렉터(`div.product-review-unit-main` 등)는 사이트 구조가 바뀌면 깨질 수 있어요.** 실행 전 실제 페이지의 HTML 구조와 한 번 비교해보는 걸 권장합니다.
- KR 크롤러와 달리 별도 로그인/인증 토큰 없이 **공개된 리뷰 화면만 읽어오는 방식**입니다.
- 클릭·요청 사이에 `time.sleep()`으로 간격을 두어, 짧은 시간에 과도한 요청이 몰리지 않도록 구성되어 있습니다.


### 수집 대상 상품 링크 예시

```
https://global.oliveyoung.com/product/detail?prdtNo=XXXXXXXXXX
```

※ `prdtNo` 값이 상품별 고유 코드입니다. 실제 상품 링크는 각 블록의 step 2 셀에서 input()으로 직접 입력받습니다.
   여러 상품을 수집하려면 상품마다 노트북(또는 해당 블록)을 반복 실행하면 됩니다.


In [ ]:
#step 1. 필요한 라이브러리 로딩
from selenium import webdriver
import undetected_chromedriver as uc
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.action_chains import ActionChains
import urllib.request, urllib
import pandas as pd
from bs4 import BeautifulSoup
import os, tempfile, time, math, random, re
from datetime import datetime, timedelta
import unicodedata
import json

- 수집 url 입력 및 파일 경로 설정

In [ ]:
# 상품 링크 입력 + 기본 저장 경로: 현재 작업 폴더 하위 output/review_gb/ (Newest 블록)
# step 2. 파일 저장경로 설정 및 url입력받기

# url
url = input('고객 후기를 수집할 제품의 링크를 입력하세요.')
print(' - 후기 수집할 제품 링크 :', url) 
print('='*60)

# goods_no
from urllib.parse import urlparse, parse_qs
parsed = urlparse(url)
goods_no = parse_qs(parsed.query).get('prdtNo', [None])[0]
print(f" - 상품 번호 : {goods_no}")
print('='*60)

# 파일 저장경로
web_name = 'oliveyoung_global'

f_dir = input('파일 저장 경로를 입력하세요. (엔터 시 기본 경로) ')
if f_dir == '':
    f_dir = "./output/\\review_gb\\"
if not f_dir.endswith('\\'):
    f_dir += '\\'

print(' - 파일 저장 경로 :', f_dir)
print('='*60)

# 시간 포함 폴더 생성
n = time.localtime()
s = '%04d%02d%02d_%02d%02d%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

folder_name = f'{s}_{web_name}_{goods_no}'
full_dir = f_dir + folder_name
os.makedirs(full_dir, exist_ok=True)
os.chdir(full_dir)

# 파일명 설정
ff_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.txt'
fc_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.csv'
fx_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.xlsx'

print(f' - 폴더명 : {folder_name}')
print('='*60)

- 크롬 드라이버 실행 상품 url 접속

In [ ]:
#실행중인 크롬창 모두 종료
import subprocess
subprocess.run(['taskkill', '/f', '/im', 'chrome.exe'], capture_output=True)
subprocess.run(['taskkill', '/f', '/im', 'chromedriver.exe'], capture_output=True)
time.sleep(2)

options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-blink-features=AutomationControlled')

user_data = os.path.join(tempfile.gettempdir(), 'us_profile_global')
os.makedirs(user_data, exist_ok=True)

CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
VERSION_MAIN = 147  # 본인 로컬 Chrome 버전에 맞게 수정 필요 (chrome://settings/help 에서 확인)

driver = uc.Chrome(
    options=options,
    browser_executable_path=CHROME_PATH,
    version_main=VERSION_MAIN,
    user_data_dir=user_data,
    use_subprocess=True,
    patcher_force_close=True,
    debug=True,
)

In [ ]:
#상품 url 접속
driver.get(url)
time.sleep(5)
driver.maximize_window()
print(driver.current_url)
print(driver.title)

- 리뷰섹션 로딩 및 수집할 리뷰건수 입력

In [ ]:
# 리뷰 버튼 클릭
review_btn = driver.find_element(By.CSS_SELECTOR, 'a[data-testid="product-review-link"]')
driver.execute_script("arguments[0].click();", review_btn)
time.sleep(2)
print('===리뷰 섹션으로 이동합니다.===')

# 총 리뷰 수 가져오기
total_review = driver.find_element(By.CSS_SELECTOR, 'p.review-list-count span').text
total_review = int(total_review.replace(',', ''))
print(f'총 리뷰 수 : {total_review}건')
print('='*40)

# 수집할 건수 입력
cnt = input(f'수집할 리뷰 건수를 입력하세요. (최대 {total_review}건) : ')
cnt = int(cnt)
if cnt > total_review:
    cnt = total_review
    
print(f'총 {total_review}건 중 {cnt}건의 리뷰를 수집합니다.')
print('='*40)

In [ ]:
# 정렬 기준: 최신순(Newest) — 다른 정렬을 원하면 아래 CSS 셀렉터를 바꾸면 됩니다.
# 정렬 버튼 클릭 (Most Helpful 버튼)
sort_btn = driver.find_element(By.CSS_SELECTOR, 'button[data-attr="PD_reviewSort_click"]')
driver.execute_script("arguments[0].click();", sort_btn)
time.sleep(1)

# Newest 클릭
newest_btn = driver.find_element(By.CSS_SELECTOR, 'li.PD_reviewSortNewest_click')
driver.execute_script("arguments[0].click();", newest_btn)
time.sleep(2)
print('===최신순으로 정렬 완료===')

- 초기화 및 리스트 정의

In [ ]:
# 수집 결과를 담을 리스트 초기화 (더보기 버튼 반복 클릭으로 로드되는 리뷰를 계속 누적)
from bs4 import BeautifulSoup

count = 0
seen_url = set()

index2 = []
goods_no2 = []
writer2 = []
skin_type2 = []
skin_tone2 = []
skin_trouble2 = []
rating2 = []
review_date2 = []
review_txt2 = []
helpful_cnt2 = []

- 리뷰 수집

In [ ]:
# '더보기' 버튼을 반복 클릭하며 새 리뷰가 없을 때까지 수집하는 메인 루프
f = open(ff_name, 'a', encoding='UTF-8')

while count < cnt:
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    reviews = soup.select('div.product-review-unit-main')

    new_reviews = 0  # 이번 루프에서 새로 추가된 리뷰 수

    for review in reviews:
        if count >= cnt:
            break

        # 리뷰 텍스트 (중복 체크용)
        review_txt_el = review.select_one('div.review-unit-cont-comment')
        review_txt = review_txt_el.text.strip() if review_txt_el else None

        if not review_txt or review_txt in seen_url:
            continue

        seen_url.add(review_txt)
        new_reviews += 1
        count += 1

        # 별점
        stars = review.select('div.wrap-icon-star')
        rating = sum(1 for s in stars if s.select_one('div.icon-star.coral-50.left.filled'))

        # 작성일자
        review_date = review.select_one('span.review-write-info-date')
        review_date = review_date.text.replace('/', '.') if review_date else None

        # 작성자
        writer = review.select_one('span.review-write-info-writer')
        writer = writer.text.replace('by.', '').strip() if writer else None

        # 피부 정보
        skin_data = {'Skin Tone': None, 'Skin Type': None, 'Skin Concern': None}
        dl_list = review.select('div.user-skin-data dl')
        for dl in dl_list:
            dt = dl.select_one('dt')
            dd_list = dl.select('dd')
            if dt and dd_list:
                key = dt.text.strip()
                value = ', '.join([dd.text.strip() for dd in dd_list])
                if key in skin_data:
                    skin_data[key] = value

        skin_tone    = skin_data['Skin Tone']
        skin_type    = skin_data['Skin Type']
        skin_trouble = skin_data['Skin Concern']

        # 추천수
        helpful_cnt = review.select_one('span.btn-likey-count')
        helpful_cnt = int(helpful_cnt.text.strip()) if helpful_cnt else 0

        print(f'{count}. {writer} | 별점:{rating} | {review_date}')

        index2.append(count)
        goods_no2.append(goods_no)
        writer2.append(writer)
        skin_type2.append(skin_type)
        skin_tone2.append(skin_tone)
        skin_trouble2.append(skin_trouble)
        rating2.append(rating)
        review_date2.append(review_date)
        review_txt2.append(review_txt)
        helpful_cnt2.append(helpful_cnt)

        f.write(f'[{count}] 작성자:{writer} | 별점:{rating} | 작성일:{review_date}\n')
        f.write(f'Skin Tone:{skin_tone} | Skin Type:{skin_type} | Skin Concern:{skin_trouble}\n')
        f.write(f'리뷰:{review_txt}\n')
        f.write(f'추천수:{helpful_cnt}\n\n')

    # 새로 추가된 리뷰가 없으면 더보기 버튼 클릭
    if new_reviews == 0 or count < cnt:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'button.review-list-more-btn')
            driver.execute_script("arguments[0].click();", more_btn)
            print(f'===더보기 버튼 클릭 ({count}개 수집 완료)===')
            time.sleep(2)
        except:
            print('더보기 버튼이 없습니다. 수집 종료.')
            break

f.close()
print(f'총 {count}개 수집 완료')
print('='*80)

- 파일 저장

In [ ]:
# 수집한 리뷰 리스트를 DataFrame으로 정리 후 CSV(+XLSX)로 저장
# step 6. csv, xlsx 형태로 저장하기
oliveyoung_global = pd.DataFrame()
oliveyoung_global['index']        = pd.Series(index2)
oliveyoung_global['goods_no']     = pd.Series(goods_no2)
oliveyoung_global['writer']       = pd.Series(writer2)
oliveyoung_global['skin_type']    = pd.Series(skin_type2)
oliveyoung_global['skin_tone']    = pd.Series(skin_tone2)
oliveyoung_global['skin_trouble'] = pd.Series(skin_trouble2)
oliveyoung_global['rating']       = pd.Series(rating2)
oliveyoung_global['review_date']  = pd.Series(review_date2)
oliveyoung_global['review_txt']   = pd.Series(review_txt2)
oliveyoung_global['helpful_cnt']  = pd.Series(helpful_cnt2)

oliveyoung_global.to_csv(fc_name, encoding='utf-8-sig', index=False)
oliveyoung_global.to_excel(fx_name, index=False, engine='openpyxl')

print('\n')
print('='*100)
print('1. 파일 저장 완료 txt  : %s' % ff_name)
print('2. 파일 저장 완료 csv  : %s' % fc_name)
print('3. 파일 저장 완료 xlsx : %s' % fx_name)
print('='*100)

In [ ]:
# 실험용 테스트 셀 - 리뷰 3개만 파싱해보는 디버깅용 (최종 수집 로직에는 미사용)
#데이터 파싱 테스트

# 현재 페이지 HTML 가져오기
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 리뷰 목록 가져오기
reviews = soup.select('div.product-review-unit-main')

# 3개만 테스트
for i, review in enumerate(reviews[:3]):
    # 별점
    stars = review.select('div.wrap-icon-star')
    rating = sum(1 for s in stars if s.select_one('div.icon-star.coral-50.left.filled'))
    
    # 작성일자
    review_date = review.select_one('span.review-write-info-date')
    review_date = review_date.text.replace('/', '.') if review_date else None
    
    # 작성자
    writer = review.select_one('span.review-write-info-writer')
    writer = writer.text.replace('by.', '').strip() if writer else None
    
    # 피부 정보
    skin_data = {'Skin Tone': None, 'Skin Type': None, 'Skin Concern': None}
    dl_list = review.select('div.user-skin-data dl')
    for dl in dl_list:
        dt = dl.select_one('dt')
        dd_list = dl.select('dd')
        if dt and dd_list:
            key = dt.text.strip()
            value = ', '.join([dd.text.strip() for dd in dd_list])
            if key in skin_data:
                skin_data[key] = value
    
    skin_tone    = skin_data['Skin Tone']
    skin_type    = skin_data['Skin Type']
    skin_trouble = skin_data['Skin Concern']
    
    # 리뷰 내용
    review_txt = review.select_one('div.review-unit-cont-comment')
    review_txt = review_txt.text.strip() if review_txt else None
    
    # 추천수
    helpful_cnt = review.select_one('span.btn-likey-count')
    helpful_cnt = int(helpful_cnt.text.strip()) if helpful_cnt else 0

    print(f'[{i+1}] 작성자:{writer} | 별점:{rating} | 작성일:{review_date}')
    print(f'     Skin Tone:{skin_tone} | Skin Type:{skin_type} | Skin Concern:{skin_trouble}')
    print(f'     리뷰:{review_txt}')
    print(f'     추천수:{helpful_cnt}')
    print()

### oliveyoung review web crawler - global (Highest rating)

In [ ]:
#step 1. 필요한 라이브러리 로딩
from selenium import webdriver
import undetected_chromedriver as uc
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.action_chains import ActionChains
import urllib.request, urllib
import pandas as pd
from bs4 import BeautifulSoup
import os, tempfile, time, math, random, re
from datetime import datetime, timedelta
import unicodedata
import json

In [ ]:
# 상품 링크 입력 + 기본 저장 경로: 현재 작업 폴더 하위 output/gb_desc/ (Highest rating 블록)
# step 2. 파일 저장경로 설정 및 url입력받기

# url
url = input('고객 후기를 수집할 제품의 링크를 입력하세요.')
print(' - 후기 수집할 제품 링크 :', url) 
print('='*60)

# goods_no
from urllib.parse import urlparse, parse_qs
parsed = urlparse(url)
goods_no = parse_qs(parsed.query).get('prdtNo', [None])[0]
print(f" - 상품 번호 : {goods_no}")
print('='*60)

# 파일 저장경로
web_name = 'gb'

f_dir = input('파일 저장 경로를 입력하세요. (엔터 시 기본 경로) ')
if f_dir == '':
    f_dir = os.path.join(os.getcwd(), "output", "gb_desc") + os.sep   
if not f_dir.endswith('\\'):
    f_dir += '\\'

print(' - 파일 저장 경로 :', f_dir)
print('='*60)

# 시간 포함 폴더 생성
n = time.localtime()
s = '%04d%02d%02d_%02d%02d%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

folder_name = f'{s}_{web_name}_{goods_no}'
full_dir = f_dir + folder_name
os.makedirs(full_dir, exist_ok=True)
os.chdir(full_dir)

# 파일명 설정
ff_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.txt'
fc_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.csv'
#fx_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.xlsx'

print(f' - 폴더명 : {folder_name}')
print('='*60)

In [ ]:
#실행중인 크롬창 모두 종료
import subprocess
subprocess.run(['taskkill', '/f', '/im', 'chrome.exe'], capture_output=True)
subprocess.run(['taskkill', '/f', '/im', 'chromedriver.exe'], capture_output=True)
time.sleep(2)

options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-blink-features=AutomationControlled')

user_data = os.path.join(tempfile.gettempdir(), 'us_profile_global')
os.makedirs(user_data, exist_ok=True)

CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
VERSION_MAIN = 147  # 본인 로컬 Chrome 버전에 맞게 수정 필요 (chrome://settings/help 에서 확인)

driver = uc.Chrome(
    options=options,
    browser_executable_path=CHROME_PATH,
    version_main=VERSION_MAIN,
    user_data_dir=user_data,
    use_subprocess=True,
    patcher_force_close=True,
    debug=True,
)

In [ ]:
# 정렬 기준: 평점 높은순(Highest rating) — 정렬 버튼 클릭 셀렉터만 다른 블록과 차이가 있어요.
#상품 url 접속
driver.get(url)
time.sleep(5)
driver.maximize_window()
print(driver.current_url)
print(driver.title)

# 리뷰 버튼 클릭
review_btn = driver.find_element(By.CSS_SELECTOR, 'a[data-testid="product-review-link"]')
driver.execute_script("arguments[0].click();", review_btn)
time.sleep(2)
print('===리뷰 섹션으로 이동합니다.===')

# 총 리뷰 수 가져오기
total_review = driver.find_element(By.CSS_SELECTOR, 'p.review-list-count span').text
total_review = int(total_review.replace(',', ''))
print(f'총 리뷰 수 : {total_review}건')
print('='*40)

# 수집할 건수 입력
cnt = input(f'수집할 리뷰 건수를 입력하세요. (최대 {total_review}건) : ')
cnt = int(cnt)
if cnt > total_review:
    cnt = total_review
    
print(f'총 {total_review}건 중 {cnt}건의 리뷰를 수집합니다.')
print('='*40)

# 정렬 버튼 클릭 (Most Helpful 버튼)
sort_btn = driver.find_element(By.CSS_SELECTOR, 'button[data-attr="PD_reviewSort_click"]')
driver.execute_script("arguments[0].click();", sort_btn)
time.sleep(1)

# Newest 클릭
newest_btn = driver.find_element(By.CSS_SELECTOR, 'li.PD_reviewSortHighestRating_click')
driver.execute_script("arguments[0].click();", newest_btn)
time.sleep(2)
print('===Highest ranking순으로 정렬 완료===')

In [ ]:
# 리스트 초기화 + '더보기' 버튼을 반복 클릭하며 새 리뷰가 없을 때까지 수집하는 메인 루프
from bs4 import BeautifulSoup

count = 0
seen_url = set()

index2 = []
goods_no2 = []
writer2 = []
skin_type2 = []
skin_tone2 = []
skin_trouble2 = []
rating2 = []
review_date2 = []
review_txt2 = []
helpful_cnt2 = []

f = open(ff_name, 'a', encoding='UTF-8')

while count < cnt:
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    reviews = soup.select('div.product-review-unit-main')

    new_reviews = 0  # 이번 루프에서 새로 추가된 리뷰 수

    for review in reviews:
        if count >= cnt:
            break

        # 리뷰 텍스트 (중복 체크용)
        review_txt_el = review.select_one('div.review-unit-cont-comment')
        review_txt = review_txt_el.text.strip() if review_txt_el else None

        if not review_txt or review_txt in seen_url:
            continue

        seen_url.add(review_txt)
        new_reviews += 1
        count += 1

        # 별점
        stars = review.select('div.wrap-icon-star')
        rating = sum(1 for s in stars if s.select_one('div.icon-star.coral-50.left.filled'))

        # 작성일자
        review_date = review.select_one('span.review-write-info-date')
        review_date = review_date.text.replace('/', '.') if review_date else None

        # 작성자
        writer = review.select_one('span.review-write-info-writer')
        writer = writer.text.replace('by.', '').strip() if writer else None

        # 피부 정보
        skin_data = {'Skin Tone': None, 'Skin Type': None, 'Skin Concern': None}
        dl_list = review.select('div.user-skin-data dl')
        for dl in dl_list:
            dt = dl.select_one('dt')
            dd_list = dl.select('dd')
            if dt and dd_list:
                key = dt.text.strip()
                value = ', '.join([dd.text.strip() for dd in dd_list])
                if key in skin_data:
                    skin_data[key] = value

        skin_tone    = skin_data['Skin Tone']
        skin_type    = skin_data['Skin Type']
        skin_trouble = skin_data['Skin Concern']

        # 추천수
        helpful_cnt = review.select_one('span.btn-likey-count')
        helpful_cnt = int(helpful_cnt.text.strip()) if helpful_cnt else 0

        print(f'{count}. {writer} | 별점:{rating} | {review_date}')

        index2.append(count)
        goods_no2.append(goods_no)
        writer2.append(writer)
        skin_type2.append(skin_type)
        skin_tone2.append(skin_tone)
        skin_trouble2.append(skin_trouble)
        rating2.append(rating)
        review_date2.append(review_date)
        review_txt2.append(review_txt)
        helpful_cnt2.append(helpful_cnt)

        f.write(f'[{count}] 작성자:{writer} | 별점:{rating} | 작성일:{review_date}\n')
        f.write(f'Skin Tone:{skin_tone} | Skin Type:{skin_type} | Skin Concern:{skin_trouble}\n')
        f.write(f'리뷰:{review_txt}\n')
        f.write(f'추천수:{helpful_cnt}\n\n')

    # 새로 추가된 리뷰가 없으면 더보기 버튼 클릭
    if new_reviews == 0 or count < cnt:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'button.review-list-more-btn')
            driver.execute_script("arguments[0].click();", more_btn)
            print(f'===더보기 버튼 클릭 ({count}개 수집 완료)===')
            time.sleep(2)
        except:
            print('더보기 버튼이 없습니다. 수집 종료.')
            break

f.close()
print(f'총 {count}개 수집 완료')
print('='*80)

In [ ]:
# 수집한 리뷰 리스트를 DataFrame으로 정리 후 CSV(+XLSX)로 저장
# step 6. csv, xlsx 형태로 저장하기
oliveyoung_global = pd.DataFrame()
oliveyoung_global['index']        = pd.Series(index2)
oliveyoung_global['goods_no']     = pd.Series(goods_no2)
oliveyoung_global['writer']       = pd.Series(writer2)
oliveyoung_global['skin_type']    = pd.Series(skin_type2)
oliveyoung_global['skin_tone']    = pd.Series(skin_tone2)
oliveyoung_global['skin_trouble'] = pd.Series(skin_trouble2)
oliveyoung_global['rating']       = pd.Series(rating2)
oliveyoung_global['review_date']  = pd.Series(review_date2)
oliveyoung_global['review_txt']   = pd.Series(review_txt2)
oliveyoung_global['helpful_cnt']  = pd.Series(helpful_cnt2)

oliveyoung_global.to_csv(fc_name, encoding='utf-8-sig', index=False)
#oliveyoung_global.to_excel(fx_name, index=False, engine='openpyxl')

print('\n')
print('='*100)
print('1. 파일 저장 완료 txt  : %s' % ff_name)
print('2. 파일 저장 완료 csv  : %s' % fc_name)
#print('3. 파일 저장 완료 xlsx : %s' % fx_name)
print('='*100)

### oliveyoung review web crawler - global (Lowest rating)

In [ ]:
#step 1. 필요한 라이브러리 로딩
from selenium import webdriver
import undetected_chromedriver as uc
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.action_chains import ActionChains
import urllib.request, urllib
import pandas as pd
from bs4 import BeautifulSoup
import os, tempfile, time, math, random, re
from datetime import datetime, timedelta
import unicodedata
import json

In [ ]:
# 상품 링크 입력 + 기본 저장 경로: 현재 작업 폴더 하위 output/gb_asc/ (Lowest rating 블록)
# step 2. 파일 저장경로 설정 및 url입력받기

# url
url = input('고객 후기를 수집할 제품의 링크를 입력하세요.')
print(' - 후기 수집할 제품 링크 :', url) 
print('='*60)

# goods_no
from urllib.parse import urlparse, parse_qs
parsed = urlparse(url)
goods_no = parse_qs(parsed.query).get('prdtNo', [None])[0]
print(f" - 상품 번호 : {goods_no}")
print('='*60)

# 파일 저장경로
web_name = 'gb'

f_dir = input('파일 저장 경로를 입력하세요. (엔터 시 기본 경로) ')
if f_dir == '':
    f_dir = os.path.join(os.getcwd(), "output", "gb_asc") + os.sep   
if not f_dir.endswith('\\'):
    f_dir += '\\'

print(' - 파일 저장 경로 :', f_dir)
print('='*60)

# 시간 포함 폴더 생성
n = time.localtime()
s = '%04d%02d%02d_%02d%02d%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

folder_name = f'{s}_{web_name}_{goods_no}'
full_dir = f_dir + folder_name
os.makedirs(full_dir, exist_ok=True)
os.chdir(full_dir)

# 파일명 설정
ff_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.txt'
fc_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.csv'
#fx_name = full_dir + '\\' + f'{s}_{web_name}_{goods_no}.xlsx'

print(f' - 폴더명 : {folder_name}')
print('='*60)

In [ ]:
#실행중인 크롬창 모두 종료
import subprocess
subprocess.run(['taskkill', '/f', '/im', 'chrome.exe'], capture_output=True)
subprocess.run(['taskkill', '/f', '/im', 'chromedriver.exe'], capture_output=True)
time.sleep(2)

options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-blink-features=AutomationControlled')

user_data = os.path.join(tempfile.gettempdir(), 'us_profile_global')
os.makedirs(user_data, exist_ok=True)

CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
VERSION_MAIN = 147  # 본인 로컬 Chrome 버전에 맞게 수정 필요 (chrome://settings/help 에서 확인)

driver = uc.Chrome(
    options=options,
    browser_executable_path=CHROME_PATH,
    version_main=VERSION_MAIN,
    user_data_dir=user_data,
    use_subprocess=True,
    patcher_force_close=True,
    debug=True,
)

In [ ]:
# 정렬 기준: 평점 낮은순(Lowest rating) — 정렬 버튼 클릭 셀렉터만 다른 블록과 차이가 있어요.
#상품 url 접속
driver.get(url)
time.sleep(5)
driver.maximize_window()
print(driver.current_url)
print(driver.title)

# 리뷰 버튼 클릭
review_btn = driver.find_element(By.CSS_SELECTOR, 'a[data-testid="product-review-link"]')
driver.execute_script("arguments[0].click();", review_btn)
time.sleep(2)
print('===리뷰 섹션으로 이동합니다.===')

# 총 리뷰 수 가져오기
total_review = driver.find_element(By.CSS_SELECTOR, 'p.review-list-count span').text
total_review = int(total_review.replace(',', ''))
print(f'총 리뷰 수 : {total_review}건')
print('='*40)

# 수집할 건수 입력
cnt = input(f'수집할 리뷰 건수를 입력하세요. (최대 {total_review}건) : ')
cnt = int(cnt)
if cnt > total_review:
    cnt = total_review
    
print(f'총 {total_review}건 중 {cnt}건의 리뷰를 수집합니다.')
print('='*40)

# 정렬 버튼 클릭 (Most Helpful 버튼)
sort_btn = driver.find_element(By.CSS_SELECTOR, 'button[data-attr="PD_reviewSort_click"]')
driver.execute_script("arguments[0].click();", sort_btn)
time.sleep(1)

# Newest 클릭
newest_btn = driver.find_element(By.CSS_SELECTOR, 'li.PD_reviewSortLowestRating_click')
driver.execute_script("arguments[0].click();", newest_btn)
time.sleep(2)
print('===Lowest ranking순으로 정렬 완료===')

In [ ]:
# 리스트 초기화 + '더보기' 버튼을 반복 클릭하며 새 리뷰가 없을 때까지 수집하는 메인 루프
from bs4 import BeautifulSoup

count = 0
seen_url = set()

index2 = []
goods_no2 = []
writer2 = []
skin_type2 = []
skin_tone2 = []
skin_trouble2 = []
rating2 = []
review_date2 = []
review_txt2 = []
helpful_cnt2 = []

f = open(ff_name, 'a', encoding='UTF-8')

while count < cnt:
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    reviews = soup.select('div.product-review-unit-main')

    new_reviews = 0  # 이번 루프에서 새로 추가된 리뷰 수

    for review in reviews:
        if count >= cnt:
            break

        # 리뷰 텍스트 (중복 체크용)
        review_txt_el = review.select_one('div.review-unit-cont-comment')
        review_txt = review_txt_el.text.strip() if review_txt_el else None

        if not review_txt or review_txt in seen_url:
            continue

        seen_url.add(review_txt)
        new_reviews += 1
        count += 1

        # 별점
        stars = review.select('div.wrap-icon-star')
        rating = sum(1 for s in stars if s.select_one('div.icon-star.coral-50.left.filled'))

        # 작성일자
        review_date = review.select_one('span.review-write-info-date')
        review_date = review_date.text.replace('/', '.') if review_date else None

        # 작성자
        writer = review.select_one('span.review-write-info-writer')
        writer = writer.text.replace('by.', '').strip() if writer else None

        # 피부 정보
        skin_data = {'Skin Tone': None, 'Skin Type': None, 'Skin Concern': None}
        dl_list = review.select('div.user-skin-data dl')
        for dl in dl_list:
            dt = dl.select_one('dt')
            dd_list = dl.select('dd')
            if dt and dd_list:
                key = dt.text.strip()
                value = ', '.join([dd.text.strip() for dd in dd_list])
                if key in skin_data:
                    skin_data[key] = value

        skin_tone    = skin_data['Skin Tone']
        skin_type    = skin_data['Skin Type']
        skin_trouble = skin_data['Skin Concern']

        # 추천수
        helpful_cnt = review.select_one('span.btn-likey-count')
        helpful_cnt = int(helpful_cnt.text.strip()) if helpful_cnt else 0

        print(f'{count}. {writer} | 별점:{rating} | {review_date}')

        index2.append(count)
        goods_no2.append(goods_no)
        writer2.append(writer)
        skin_type2.append(skin_type)
        skin_tone2.append(skin_tone)
        skin_trouble2.append(skin_trouble)
        rating2.append(rating)
        review_date2.append(review_date)
        review_txt2.append(review_txt)
        helpful_cnt2.append(helpful_cnt)

        f.write(f'[{count}] 작성자:{writer} | 별점:{rating} | 작성일:{review_date}\n')
        f.write(f'Skin Tone:{skin_tone} | Skin Type:{skin_type} | Skin Concern:{skin_trouble}\n')
        f.write(f'리뷰:{review_txt}\n')
        f.write(f'추천수:{helpful_cnt}\n\n')

    # 새로 추가된 리뷰가 없으면 더보기 버튼 클릭
    if new_reviews == 0 or count < cnt:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'button.review-list-more-btn')
            driver.execute_script("arguments[0].click();", more_btn)
            print(f'===더보기 버튼 클릭 ({count}개 수집 완료)===')
            time.sleep(2)
        except:
            print('더보기 버튼이 없습니다. 수집 종료.')
            break

f.close()
print(f'총 {count}개 수집 완료')
print('='*80)

In [ ]:
# 수집한 리뷰 리스트를 DataFrame으로 정리 후 CSV(+XLSX)로 저장
# step 6. csv, xlsx 형태로 저장하기
oliveyoung_global = pd.DataFrame()
oliveyoung_global['index']        = pd.Series(index2)
oliveyoung_global['goods_no']     = pd.Series(goods_no2)
oliveyoung_global['writer']       = pd.Series(writer2)
oliveyoung_global['skin_type']    = pd.Series(skin_type2)
oliveyoung_global['skin_tone']    = pd.Series(skin_tone2)
oliveyoung_global['skin_trouble'] = pd.Series(skin_trouble2)
oliveyoung_global['rating']       = pd.Series(rating2)
oliveyoung_global['review_date']  = pd.Series(review_date2)
oliveyoung_global['review_txt']   = pd.Series(review_txt2)
oliveyoung_global['helpful_cnt']  = pd.Series(helpful_cnt2)

oliveyoung_global.to_csv(fc_name, encoding='utf-8-sig', index=False)
#oliveyoung_global.to_excel(fx_name, index=False, engine='openpyxl')

print('\n')
print('='*100)
print('1. 파일 저장 완료 txt  : %s' % ff_name)
print('2. 파일 저장 완료 csv  : %s' % fc_name)
#print('3. 파일 저장 완료 xlsx : %s' % fx_name)
print('='*100)